<a href="https://colab.research.google.com/github/srushtitelang18/Satellite-Land-Cover-Classification/blob/main/satellite_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio transformers torch torchvision pillow matplotlib opencv-python

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import cv2
from PIL import Image
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

model_name = "nvidia/segformer-b5-finetuned-ade-640-640"

processor = SegformerImageProcessor.from_pretrained(model_name)
model = SegformerForSemanticSegmentation.from_pretrained(model_name).to(DEVICE)
model.eval()

print("✅ High-accuracy model loaded")

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/339M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/339M [00:00<?, ?B/s]

✅ High-accuracy model loaded


In [ ]:
URBAN_CLASSES = {
    "building": [1, 2, 13, 25],
    "road": [3, 6, 11, 52],
    "vegetation": [4, 9, 17, 29],
    "water": [21, 26, 60]
}

COLORS = {
    "building": (255, 140, 0),
    "road": (138, 43, 226),
    "vegetation": (34, 139, 34),
    "water": (30, 144, 255),
    "other": (50, 50, 50)
}

In [ ]:
def segment_image(image):
    image = image.convert("RGB")

    # 🔥 HIGH RESOLUTION (important)
    image = image.resize((640, 640))

    inputs = processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    upsampled = F.interpolate(
        logits,
        size=image.size[::-1],
        mode="bilinear",
        align_corners=False
    )

    pred = upsampled.argmax(dim=1)[0].cpu().numpy()

    h, w = pred.shape
    mask = np.zeros((h, w, 3), dtype=np.uint8)

    for cls, indices in URBAN_CLASSES.items():
        for idx in indices:
            mask[pred == idx] = COLORS[cls]

    mask[(mask == 0).all(axis=2)] = COLORS["other"]

    # 🔥 POST-PROCESSING (important)
    mask = cv2.medianBlur(mask, 5)
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    return Image.fromarray(mask)

In [ ]:
def overlay(image, mask):
    image = image.resize((640, 640))
    image = np.array(image)
    mask = np.array(mask)

    result = (0.6 * image + 0.4 * mask).astype(np.uint8)
    return Image.fromarray(result)

In [ ]:
def process(image):
    mask = segment_image(image)
    final = overlay(image, mask)
    return mask, final

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## 🛰️ Satellite Image Segmentation System")
    gr.Markdown("Upload image → Detect Buildings, Roads, Water, Vegetation")

    with gr.Row():
        input_img = gr.Image(type="pil", label="📂 Upload Satellite Image")

    run_btn = gr.Button("🚀 Run Segmentation")

    with gr.Row():
        output_mask = gr.Image(label="🗺 Segmented Map")
        output_overlay = gr.Image(label="🌍 Overlay Output")

    run_btn.click(fn=process, inputs=input_img, outputs=[output_mask, output_overlay])

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d366bf10a913df4d33.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
